# Peptides-struct: canonical scores + carriage (15 checkpoints)

Use an **A100 80GB + high-RAM** runtime and choose **Runtime → Run all**. This notebook verifies/extracts the uploaded corpus automatically, then runs dense, 1-hop, 1-hop+VNode, 2-hop, and 2-hop+VNode for seeds 0/1/2 in sequence.

Every graph shard and consolidated cache is written directly to Drive. Rerunning validates and skips completed checkpoints, and resumes partial checkpoints. After both the func and struct notebooks reach 15/15, set `MODE = "finalize"` in either notebook and rerun the final cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODE = "run"  # @param ["run", "setup", "status", "finalize"]
DATASET = "struct"
DRIVE_FOLDER = "/content/drive/MyDrive/graph_specialisation_metrics/multi_seed_models/peptides_func_struct_checkpoints"
GRAPHS_PER_BATCH = 0  # @param {type:"integer"}
RECLAIM_WORKER_INDEX = -1  # @param {type:"integer"}
STRICT_AUDITS = False  # @param {type:"boolean"}
REPO_REVISION = "80f70a81ab703fe8aac8a7934b31f91522788ee3"

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

repo = Path('/content/Graph-Specialisation-and-Metrics')
url = 'https://github.com/joshgreenwa/Graph-Specialisation-and-Metrics.git'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REPO_REVISION], check=True)
for path in (str(repo), str(repo / 'src')):
    if path not in sys.path:
        sys.path.insert(0, path)
for name in tuple(sys.modules):
    if name in {
        'experiments.methodology.peptides_func_struct_canonical_colab',
        'experiments.methodology.zinc_qm9_canonical_colab_worker',
    } or name.startswith('graph_specialisation_metrics.'):
        del sys.modules[name]
methodology_package = sys.modules.get('experiments.methodology')
if methodology_package is not None:
    vars(methodology_package).pop('peptides_func_struct_canonical_colab', None)
    vars(methodology_package).pop('zinc_qm9_canonical_colab_worker', None)
importlib.invalidate_caches()
print(f'[repo] pinned at {REPO_REVISION}')

In [ ]:
from experiments.methodology.peptides_func_struct_canonical_colab import run_frontend

result = run_frontend(
    mode=MODE,
    dataset=DATASET,
    drive_folder=DRIVE_FOLDER,
    graphs_per_batch=GRAPHS_PER_BATCH or None,
    accelerator='cuda:0',
    strict_audits=STRICT_AUDITS,
    reclaim_worker_index=RECLAIM_WORKER_INDEX,
)